# Lab 1 — zadania domowe

Dekoratory, deskryptory i generatory.

## Zadanie 1 — dekorator `require_role`

Stwórz dekorator `@require_role(role)`, który sprawdza rolę użytkownika w słowniku `current_user`. Jeśli rola się nie zgadza, dekorator zgłasza `PermissionError`.

In [1]:
import functools

current_user = {"username": "admin", "role": "superuser"}


def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(f"Brak wymaganej roli: {role}")
            return func(*args, **kwargs)
        return wrapper
    return decorator


@require_role("superuser")
def open_admin_panel():
    return "Panel administratora został otwarty."


print(open_admin_panel())

current_user["role"] = "user"

try:
    print(open_admin_panel())
except PermissionError as error:
    print(error)

current_user["role"] = "superuser"

Panel administratora został otwarty.
Brak wymaganej roli: superuser


## Zadanie 2 — deskryptor `Typed`

Stwórz deskryptor `Typed`, który przyjmuje oczekiwany typ danych i sprawdza typ przypisywanej wartości. Jeśli typ jest niepoprawny, deskryptor zgłasza `TypeError`.

In [2]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.private_name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            expected_type_name = self.expected_type.__name__
            raise TypeError(f"Atrybut '{self.name}' musi mieć typ {expected_type_name}")
        instance.__dict__[self.private_name] = value


class Product:
    name = Typed(str)
    price = Typed(float)
    quantity = Typed(int)

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    def __str__(self):
        return f"{self.name}: {self.price} x {self.quantity}"


product = Product("Laptop", 3500.0, 2)
print(product)

try:
    product.quantity = "dwa"
except TypeError as error:
    print(error)

Laptop: 3500.0 x 2
Atrybut 'quantity' musi mieć typ int


## Zadanie 3 — generator liczb pierwszych

Opracuj nieskończony generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie utwórz iterator zwracający tylko liczby pierwsze kończące się cyfrą `7`.

In [3]:
def is_prime(number):
    if number < 2:
        return False

    for divisor in range(2, int(number ** 0.5) + 1):
        if number % divisor == 0:
            return False

    return True


def prime_generator():
    number = 2

    while True:
        if is_prime(number):
            yield number
        number += 1


prime_numbers_ending_with_7 = (
    number for number in prime_generator()
    if number % 10 == 7
)

result = []

for _ in range(10):
    result.append(next(prime_numbers_ending_with_7))

print(result)

[7, 17, 37, 47, 67, 97, 107, 127, 137, 157]
